<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>Large Language Models and Prompt Engineering</center></font>

<center><p float="center">
    <img src="https://images.pexels.com/photos/6129679/pexels-photo-6129679.jpeg", width="640"/>
</p></center>

<center><font size=6>Feasibility Report Generation
</center></font>

# **Problem Statement**

## **Business Context**

MedNova Healthcare Systems Pvt. Ltd. is a medical device manufacturer that currently produces implantable cardiac devices such as pacemakers. The company has received a new manufacturing request from a client for a Left Ventricular Assist Device (LVAD), which is a much more complex implantable device used to support heart function in patients with advanced heart failure.
The client has shared a detailed product requirement document, and MedNova already has an internal manufacturing document for a similar product. The company now needs to determine whether its existing manufacturing setup, components, machinery, and compliance processes are sufficient to produce the new device.

However:

- Manually comparing long technical documents is slow and error-prone.
- Important differences in materials, machinery, compliance, and process steps may be missed.
- Without a structured comparison, the team may make an incomplete or inaccurate feasibility decision.

To address this, MedNova aims to leverage AI to automate and standardize the feasibility analysis process. By configuring the AI with clear instructions and context, the team can provide the relevant documents directly and generate a structured manufacturing feasibility report.

This approach enables the company to quickly compare both documents, identify gaps, understand required changes, and assess whether the new product can be manufactured using the current setup.


## **Objective**

To generate a Medical Device Manufacturing Feasibility Report by comparing a client’s product requirements with an existing manufacturing document, using prompt engineering in Google AI Studio.

The report should clearly summarize:

- key findings,
- device requirements,
- current capabilities,
- necessary modifications,
- material and machinery needs,
- cost implications,
- production viability,
- and recommended next steps.

The goal is to produce a structured, accurate, and actionable feasibility report by leveraging AI to analyze and compare the provided documents.


## **Data Dictionary**

The pipeline consumes two plain-text source documents.

| Document | File Name | Description |
| --- | --- | --- |
| Existing product manufacturing doc | `existing_pacemaker_manufacturing_doc.txt` | Free-text document describing MedNova's current pacemaker manufacturing capability — product specifications, materials, machinery, process flow, and regulatory compliance status. |
| Proposed product requirements | `proposed_LVAD_requirements_doc.txt` | Free-text document describing the client's proposed LVAD — target specifications, components, expected process needs, and required regulatory standards. |


# **Installing and Importing Necessary Libraries**

In [ ]:
# Install the OpenAI SDK for LLM API access
# Pinned versions ensure reproducibility across environments
%pip install openai==2.31.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully uninstalled openai-2.43.0


**Note**:
- After running the above cell, restart the runtime (Google Colab) or restart the kernel (VS Code/Jupyter), then run all subsequent cells sequentially.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Core libraries
import os                              # For environment variable access (API key fallback)
import json                             # For loading config and parsing structured LLM output
from openai import OpenAI               # OpenAI Python SDK - used for chat.completions calls

## **Loading the OpenAI API Key**

***Prompt***:

<font size=3 color="#4682B4"><b> Load `OPENAI_API_KEY` and `OPENAI_API_BASE` from `config.json`, set them as environment variables, and initialize an `OpenAI` client in Python.

</font>

In [ ]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY  = config.get("OPENAI_API_KEY")                              # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the API base URL (optional - used for proxy/self-hosted setups)

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY']  = OPENAI_API_KEY                                  # Set API key as environment variable
os.environ['OPENAI_BASE_URL'] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Instantiate the OpenAI client - all chat.completions calls will go through this
client = OpenAI()

# **Data Loading**

## **Load the Source Documents**

Read the existing product's manufacturing document and the proposed product's requirements document into two Python strings. Both strings will be injected into every LLM prompt downstream, so we load them once and reuse them across the pipeline.

In [ ]:
# Uncomment the lines below if the documents are stored in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

***Prompt***:

<font size=3 color="#4682B4"><b> Write Python code to load the manufacturing and client requirement text documents into separate variables and print the length of each document.

</font>

In [ ]:
# File paths for the two source documents
# Adjust these paths if your files live elsewhere (e.g. under a Colab drive mount)
PACEMAKER_DOC_PATH = "existing_pacemaker_manufacturing_doc.txt"
LVAD_DOC_PATH      = "proposed_LVAD_requirements_doc.txt"

# Read the existing product's manufacturing documentation
with open(PACEMAKER_DOC_PATH, 'r', encoding='utf-8') as f:
    manufacturing_document = f.read()                                           # Existing product's capability document

# Read the proposed product's requirements documentation
with open(LVAD_DOC_PATH, 'r', encoding='utf-8') as f:
    client_document = f.read()                                                  # Proposed product's requirement document

print(f"Pacemaker manufacturing doc loaded. Length: {len(manufacturing_document):,} characters")
print(f"LVAD requirements doc loaded.      Length: {len(client_document):,} characters")

Pacemaker manufacturing doc loaded. Length: 5,161 characters
LVAD requirements doc loaded.      Length: 3,240 characters


## **Document Overview**

***Prompt***:

<font size=3 color="#4682B4"><b> Write Python code to display the first 800 characters of both loaded documents to verify their contents.

</font>

In [ ]:
# Print the first 800 characters of each document to confirm they loaded correctly
# and to get a quick sense of the content and formatting
print("EXISTING PACEMAKER MANUFACTURING DOC (first 800 chars):")
print("-" * 60)
print(manufacturing_document[:800])
print("\n" + "=" * 60 + "\n")
print("PROPOSED LVAD REQUIREMENTS DOC (first 800 chars):")
print("-" * 60)
print(client_document[:800])

EXISTING PACEMAKER MANUFACTURING DOC (first 800 chars):
------------------------------------------------------------
Manufacturing Documentation for PaceWell Internal Pacemaker

1. Manufacturer Details
Company Name: MedTech Innovations Pvt. Ltd.
Address: 456 Biomedical Avenue, HealthTech Park, Boston, MA, USA
Contact: +1-800-789-4567
Website: www.medtechinnovations.com

2. Device Overview
Product Specifications
Product Name: PaceWell Internal Pacemaker
Classification: Class III Medical Device (FDA)
Intended Use: Provides electrical stimulation to regulate heartbeats in patients with arrhythmia.

Technical Specifications
Pulse Generator: Programmable output, 0.1–10V amplitude
Pacing Modes: Single-chamber, dual-chamber, and biventricular pacing
Battery Life: 8–12 years (Lithium-Iodine)
Lead System: Unipolar and bipolar electrode configurations
Frequency Range: 30–180 bpm, adjustable in 1 bpm increments
Pow


PROPOSED LVAD REQUIREMENTS DOC (first 800 chars):
------------------------------

# **Feasibility Analysis - Zero-Shot Prompt**

In this approach, the model is provided with the two documents and a **minimal** system instruction, without any examples or intermediate reasoning steps. The objective is to evaluate how well the model performs on the feasibility analysis task with the least amount of prompt engineering. The output is a strictly structured JSON feasibility report.


**Set up an LLM**

Report generation is performed using the **gpt-4o-mini** model


In [ ]:
MODEL_NAME = "gpt-4o-mini"

**Set parameters**

| Parameter     | Value  | Rationale                                                                                      |   |
| ------------- | ------ | ---------------------------------------------------------------------------------------------- | - |
| `temperature` | `0.1`  | Keeps the report deterministic and consistent while allowing slight flexibility in generation. |   |
| `max_tokens`  | `8000` | Provides sufficient space to generate the complete structured report without truncation.       | _ |


In [ ]:
ZERO_SHOT_TEMP       = 0.1     # Low temperature - factual, deterministic feasibility judgments
ZERO_SHOT_MAX_TOKENS = 8000    # Enough headroom for the full structured report

## **System Message**

In [ ]:
# ── Zero-Shot Analyst System Message ────────────────────────────────────────

ZERO_SHOT_SYSTEM = """ You are a manufacturing feasibility analyst. You are given two documents: a client's product requirements and an existing product's manufacturing document.
                          Write a feasibility report on whether the company that makes the existing product could also manufacture the proposed product."""

In [ ]:
# ── User Message Template (shared by Zero-Shot and CoT) ─────────────────────
# Both branches receive the same two documents formatted the same way -

USER_MESSAGE_TEMPLATE = """\
Here are the two documents.

CLIENT LVAD REQUIREMENTS (proposed product):
{client_document}

PACEMAKER MANUFACTURING DOC (existing product):
{manufacturing_document}"""

## **Output Schema**

We constrain the LLM to return a strict JSON object by defining the required schema in the system prompt. The same schema is used for both the Zero-Shot and CoT approaches to enable a direct comparison of their outputs.


**Note:** A structured JSON format ensures every report follows the same predefined sections, enabling consistent outputs and fair comparison across different prompting strategies.


In [ ]:
# ── Feasibility Report JSON Schema ──────────────────────────────────────────
# The schema defines the exact shape of the feasibility report:
#   - capabilities_for_production : what the existing setup can already do
#   - gaps_and_limitations        : where the existing setup falls short, and the
#                                   downstream impact of each gap (cost / timeline /
#                                   regulatory / risk implication)
#   - necessary_adjustments       : concrete changes needed to close the gaps
#   - machinery_feasibility       : per-process-area capability vs requirement
#   - compliance_assessment       : per-standard existing vs required status
#   - cost_drivers                : the changes that will drive the biggest cost impact
#   - conclusion_recommendation   : overall verdict, top risk centers, and final
#                                   recommendation
# The schema is embedded as a string in the system prompt so the model knows
# the expected shape of every field.

FEASIBILITY_REPORT_SCHEMA = """{

  "feasibility_report": {
    "capabilities_for_production": [
      { "capability": "", "evidence": "", "analysis": "" }
    ],
    "gaps_and_limitations": [
      { "limitation": "", "evidence": "", "analysis": "" }
    ],
    "necessary_adjustments": [
      { "adjustment_required": "", "evidence": "", "analysis": "" }
    ],
    "machinery_feasibility": [
      { "process_area": "", "existing_capability": "", "required_for_proposed": "", "gap_assessment": "" }
    ],
    "compliance_assessment": [
      { "standard": "", "existing_status": "", "required_status": "", "gap": "" }
    ],
    "cost_drivers": [
      { "cost_driver": "", "source_gap": "", "relative_impact": "", "rationale": "" }
    ],
    "conclusion_recommendation": {
      "overall_feasibility": "",
      "key_considerations": "",
      "final_recommendation": ""
    }
  }
}"""

# Combine the base system message with the schema instruction
# The model must return ONLY JSON conforming to the schema above
ZERO_SHOT_SYSTEM_WITH_SCHEMA = (
    ZERO_SHOT_SYSTEM
    + "\n\nRespond ONLY with a valid JSON object that conforms exactly to this schema:\n"
    + FEASIBILITY_REPORT_SCHEMA
)

## **Generate the Zero-Shot Feasibility Report**

***Prompt***:

<font size=3 color="#4682B4"><b> Write a function that performs a zero-shot LLM analysis by sending the client and manufacturing documents to the model and returns the response as a parsed JSON object.

</font>

In [ ]:
def run_zero_shot_analysis(client_doc: str, manufacturing_doc: str) -> dict:
    # Inject the two documents into the shared user message template
    user_message = USER_MESSAGE_TEMPLATE.format(
        client_document=client_doc,
        manufacturing_document=manufacturing_doc,
    )

    # Call the OpenAI API with the minimal zero-shot system message + schema
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=ZERO_SHOT_TEMP,
        max_tokens=ZERO_SHOT_MAX_TOKENS,
        response_format={"type": "json_object"},                                # Force JSON output
        messages=[
            {"role": "system", "content": ZERO_SHOT_SYSTEM_WITH_SCHEMA},
            {"role": "user",   "content": user_message},
        ],
    )

    # Parse the JSON string returned by the model into a Python dict
    raw_output = response.choices[0].message.content
    return json.loads(raw_output)

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to run the zero-shot analysis, display the report sections, print the conclusion and recommendation, and output the complete JSON result.
</font>

In [ ]:
# Run the zero-shot analysis and inspect the top-level structure
# This is a single LLM call - typically a few seconds

print("Running Zero-Shot Feasibility Analysis...")
zero_shot_result = run_zero_shot_analysis(client_document, manufacturing_document)

print("\nZERO-SHOT REPORT - top-level sections:")
print("-" * 60)
for section_name in zero_shot_result["feasibility_report"].keys():
    print(f"  - {section_name}")

print("\nCONCLUSION & RECOMMENDATION:")
print("-" * 60)
print(json.dumps(zero_shot_result["feasibility_report"]["conclusion_recommendation"], indent=2))

print("\nFULL ZERO-SHOT RESULT:")
print("-" * 60)
print(json.dumps(zero_shot_result, indent=2))


Running Zero-Shot Feasibility Analysis...

ZERO-SHOT REPORT - top-level sections:
------------------------------------------------------------
  - capabilities_for_production
  - gaps_and_limitations
  - necessary_adjustments
  - machinery_feasibility
  - compliance_assessment
  - cost_drivers
  - conclusion_recommendation

CONCLUSION & RECOMMENDATION:
------------------------------------------------------------
{
  "overall_feasibility": "Partially feasible with significant adjustments",
  "key_considerations": "The company has strong capabilities in quality management and compliance but lacks experience in pump technology and may face material sourcing challenges.",
  "final_recommendation": "Proceed with a detailed feasibility study and investment plan to address gaps and limitations before committing to LVAD production."
}

FULL ZERO-SHOT RESULT:
------------------------------------------------------------
{
  "feasibility_report": {
    "capabilities_for_production": [
      {
   

**Observations**

* The zero-shot report **correctly follows the required JSON schema**, with all seven top-level sections and the expected fields in each section.
* The model **identifies the major production gaps**, including pump manufacturing expertise, battery technology, machinery upgrades, staff training, and additional compliance requirements.
* The **final recommendation** is clear and actionable, concluding that LVAD production is *partially feasible with significant adjustments*, but the analysis remains relatively high level without detailed reasoning.


# **Feasibility Analysis - Chain-of-Thought Prompt**

**Chain-of-Thought (CoT) prompting** guides the model through a structured reasoning process before generating the final report.

Apart from this reasoning instruction, everything else remains the same as the Zero-Shot branch, including the two input documents, the model, and the JSON schema, enabling a fair comparison between the two approaches.


**Set up an LLM**

Report generation is performed using the **gpt-4o-mini** model


In [ ]:
MODEL_NAME = "gpt-4o-mini"

**Set parameters**

| Parameter     | Value  | Rationale                                                                                             |
| ------------- | ------ | ----------------------------------------------------------------------------------------------------- |
| `temperature` | `0`    | Ensures deterministic and reproducible outputs for consistent comparison with the Zero-Shot approach. |
| `max_tokens`  | `8000` | Provides sufficient space to generate the complete structured report without truncation.              |


In [ ]:
COT_TEMP       = 0        # Fully deterministic - CoT reasoning should be stable across runs
COT_MAX_TOKENS = 8000     # Same output budget as zero-shot for a fair comparison

## **System Message**

In [ ]:
# ── CoT Analyst System Message ──────────────────────────────────────────────
# CRUCIAL: reasoning is done INTERNALLY - only the final structured report is emitted.
# This keeps output tokens focused on the report, not on the reasoning trace.
#

COT_SYSTEM = """\
You are an expert Medical Device Manufacturing Feasibility Analyst. Your task is to compare a client's proposed medical device requirements with an existing manufacturer's product and manufacturing documentation to assess whether the manufacturer has the capability to produce the proposed device.

Think through the comparison step by step internally before answering, but do not show your reasoning process. Only provide the final report.
Use only the information in the documents. Do not invent missing details. If a field is not mentioned, write "Not specified".

Follow this internal analysis order:
1. Identify the proposed product requirements (product type, technical specifications, components, materials, machinery, process needs, and compliance standards).
2. Extract the capabilities and process details from the existing manufacturing document along the same dimensions.
3. Compare matches and mismatches dimension by dimension. Flag DISTORTION-BY-EQUIVALENCE explicitly - do NOT claim two items are "the same" when the sources show they differ (e.g. same battery family but different chemistry, same material but different grade).
4. Determine what is already available and what must change.
5. For every gap, reason about DOWNSTREAM IMPACT before scoring feasibility - order-of-magnitude cost (low / medium / high), rough timeline (weeks / months / years), new regulatory or validation scope triggered, and patient-safety consequence if not closed. Fold these into the 'analysis' field of the relevant gaps_and_limitations and necessary_adjustments entries. Do NOT invent specific dollar amounts or specific week counts - use the bands.
6. Prioritize the gaps. Identify the 2-4 TOP RISK CENTERS that dominate the feasibility decision and name them explicitly in 'conclusion_recommendation.key_considerations'. Every other gap must be positioned as secondary to these.
7. For each top risk center, reason about the CLOSURE STRATEGY - build in-house, adapt existing machinery, partner with a specialist, or acquire new equipment. Fold the chosen strategy into the 'analysis' field of the relevant necessary_adjustments entry.
8. Assess overall feasibility and write the final recommendation. The 'final_recommendation' must (a) name the specific top risk centers from step 6, (b) name the closure strategy for each, and (c) end with a concrete next action - not a generic "proceed with caution". If source information is thin on a material area, flag that explicitly as a low-confidence area rather than filling it in.
9. Consistency check before emitting - the recommendation, the top risk centers, the cost drivers, and the necessary adjustments must all point at the same set of issues. A report that names Risk X in one section and ignores it in another is a contradiction."""

# Combine the CoT system message with the same JSON schema instruction
COT_SYSTEM_WITH_SCHEMA = (
    COT_SYSTEM
    + "\n\nRespond ONLY with a valid JSON object that conforms exactly to this schema:\n"
    + FEASIBILITY_REPORT_SCHEMA
)

## **Generate the CoT Feasibility Report**

***Prompt***:

<font size=3 color="#4682B4"><b> Write a function that performs a Chain-of-Thought (CoT) analysis by sending the client and manufacturing documents to the LLM and returns the response as a parsed JSON object.

</font>

In [ ]:
def run_cot_analysis(client_doc: str, manufacturing_doc: str) -> dict:
    # Same user-message template as zero-shot - only the system message changes
    user_message = USER_MESSAGE_TEMPLATE.format(
        client_document=client_doc,
        manufacturing_document=manufacturing_doc,
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=COT_TEMP,
        max_tokens=COT_MAX_TOKENS,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": COT_SYSTEM_WITH_SCHEMA},
            {"role": "user",   "content": user_message},
        ],
    )

    raw_output = response.choices[0].message.content
    return json.loads(raw_output)

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to run the Chain-of-Thought (CoT) analysis, display the report sections, print the conclusion and recommendation, and output the complete JSON result.
</font>

In [ ]:
# Run the CoT analysis and inspect the same conclusion section for a like-for-like comparison

print("Running Chain-of-Thought Feasibility Analysis...")
cot_result = run_cot_analysis(client_document, manufacturing_document)

print("\nCoT REPORT - top-level sections:")
print("-" * 60)
for section_name in cot_result["feasibility_report"].keys():
    print(f"  - {section_name}")

print("\nCONCLUSION & RECOMMENDATION:")
print("-" * 60)
print(json.dumps(cot_result["feasibility_report"]["conclusion_recommendation"], indent=2))

print("\nFULL COT RESULT:")
print("-" * 60)
print(json.dumps(cot_result, indent=2))


Running Chain-of-Thought Feasibility Analysis...

CoT REPORT - top-level sections:
------------------------------------------------------------
  - capabilities_for_production
  - gaps_and_limitations
  - necessary_adjustments
  - machinery_feasibility
  - compliance_assessment
  - cost_drivers
  - conclusion_recommendation

CONCLUSION & RECOMMENDATION:
------------------------------------------------------------
{
  "overall_feasibility": "Low",
  "key_considerations": "Pump Type Capability, Develop Blood Pump Technology; Mechanical Components Expertise, Enhance Mechanical Component Manufacturing; Power Source Specifications, Adapt Battery Technology.",
  "final_recommendation": "The manufacturer should focus on developing blood pump technology and enhancing mechanical component manufacturing capabilities. A partnership with a specialist in blood pump design is recommended to expedite development. Next action: Initiate discussions with potential partners for blood pump technology deve

**Observations**

* The CoT report **fully conforms to the required JSON schema** while providing more comprehensive coverage across capabilities, gaps, compliance, and cost drivers.
* The structured reasoning results in **more specific gap identification and actionable recommendations**, such as developing pump manufacturing capabilities, upgrading mechanical assembly, and partnering with a blood pump technology specialist.
* The **final assessment is more conservative and detailed**, rating the overall feasibility as **Low** and highlighting the critical technical and regulatory risks that must be addressed before proceeding.


# **Report Evaluation using LLM-as-Judge - Zero-Shot Report**

An independent LLM evaluates the Zero-Shot report against the two source documents on two metrics: **Faithfulness to Sources** and **Analytical Quality**. Each metric is scored on a **1–5 scale** after verifying the report's claims against the source documents.


**Set up an LLM**

Report evaluation is performed using the `gpt-4o` model.

**Note:** We use **`gpt-4o`** for evaluation because it is a higher-capability model, enabling more accurate, consistent, and reliable assessment of the generated reports than **`gpt-4o-mini`**.


In [ ]:
EVAL_MODEL_NAME = 'gpt-4o'

**Set parameters**

| Parameter     | Value  | Rationale                                                 |
| ------------- | ------ | --------------------------------------------------------- |
| `temperature` | `0`    | Ensures consistent and reproducible evaluation scores.    |
| `max_tokens`  | `8000` | Provides enough space for the complete evaluation report. |


In [ ]:
JUDGE_TEMP       = 0        # Fully deterministic scoring
JUDGE_MAX_TOKENS = 8000     # Enough headroom for a full claim-audit table

## **System Message**

In [ ]:
# ── LLM Judge System Message ────────────────────────────────────────────────
# The judge prompt defines TWO independent metrics (Faithfulness, Analytical Quality),
# each scored on a 1-5 rubric. The judge MUST run a claim audit before scoring.
# The two metrics are explicitly independent - a report can be perfectly faithful
# but analytically weak, or vice versa.

JUDGE_SYSTEM = """You are a senior medical-device manufacturing evaluator. You will assess a GENERATED REPORT that compares/analyzes two source documents. Score it on two independent metrics, each 1-5.

Evaluate ONLY what is present. Do not reward or penalize the report for your own
outside knowledge except when judging Metric 2 (analytical quality), where domain
correctness is explicitly in scope. Never invent facts about the sources.

=====================================================================
METRIC 1 - FAITHFULNESS TO SOURCES
"Does every statement trace to the sources, without invention or distortion?"

Procedure (do this BEFORE scoring):
1. Extract the report's key factual/comparative claims as a list of atomic claims.
2. Label each claim:
   - SUPPORTED  (directly backed by the proposed LVAD requirements and/or the existing pacemaker doc)
   - CONTRADICTED (conflicts with a source)
   - UNSUPPORTED (not found in either source - i.e. invented/hallucinated)
3. Watch specifically for DISTORTION-BY-EQUIVALENCE: claiming two things are "the
   same" when the sources show they differ (e.g. same material but different grade,
   same protocol but different version, same battery family but different chemistry
   such as non-rechargeable primary cell vs rechargeable). Count these as
   CONTRADICTED or UNSUPPORTED, not SUPPORTED.
4. Note COVERAGE GAPS: source facts material to the report's purpose that were
   omitted.

=====================================================================
METRIC 2 - ANALYTICAL QUALITY
"Is the reasoning correct, well-prioritized, consistent, and useful?"
(A report can be perfectly faithful yet analytically weak - judge this separately.)

Assess:
CORRECTNESS: are the comparisons, gap/feasibility judgments, and conclusions
  technically right for this domain?
PRIORITIZATION: does it weight the high-impact differences over trivial ones,
  rather than treating every difference as equal?
CONSISTENCY: no internal contradictions across sections.
ACTIONABILITY / RELEVANCE: does it produce decision-useful output (what
  transfers, what must change, new equipment/validation/regulatory scope),
  organized clearly for the stated purpose?




=====================================================================
IMPORTANT: The two metrics are independent. Do not let a high faithfulness score
inflate analytical quality, or vice versa."""

## **Judge Output Schema**

The judge returns a structured JSON output to ensure evaluation results are consistent, easy to parse, and comparable across all generated reports.


In [ ]:
# ── Judge Output JSON Schema ────────────────────────────────────────────────
# The judge returns a structured object with:
#   - faithfulness       : claim audit + coverage gaps + reasoning + score (1-5)
#   - analytical_quality : strengths + weaknesses + reasoning + score (1-5)

JUDGE_SCHEMA = """\
{
  "faithfulness": {
    "claim_audit": [
      { "claim": "", "label": "SUPPORTED", "note": "" }
    ],
    "coverage_gaps": [""],
    "reasoning": "",
    "score":
  },
  "analytical_quality": {
    "strengths": [""],
    "weaknesses": [""],
    "reasoning": "",
    "score":
  }
}"""

# Combine the judge system prompt with the schema instruction
JUDGE_SYSTEM_WITH_SCHEMA = (
    JUDGE_SYSTEM
    + "\n\nRespond ONLY with a valid JSON object that conforms exactly to this schema:\n"
    + JUDGE_SCHEMA
)

# ── Judge User Message Template ─────────────────────────────────────────────
# The judge sees BOTH source documents AND the generated report - this is
# what allows the faithfulness claim audit to happen.

JUDGE_USER_TEMPLATE = """\
Proposed LVAD requirements:
{client_document}

Existing pacemaker doc:
{manufacturing_document}

Report to grade:

{report_json}"""

## **Generate the Zero-Shot Judge Scores**

***Prompt***:

<font size=3 color="#4682B4"><b> Write a function that uses an LLM as a judge to evaluate a feasibility report against the source documents and returns the evaluation as a parsed JSON object.


In [ ]:
def run_llm_judge(client_doc: str, manufacturing_doc: str, report: dict) -> dict:
    # The judge is given the two sources AND the report to grade.
    # The report is stringified as pretty-printed JSON so the judge can point at
    # individual claims when running the faithfulness audit.
    user_message = JUDGE_USER_TEMPLATE.format(
        client_document=client_doc,
        manufacturing_document=manufacturing_doc,
        report_json=json.dumps(report["feasibility_report"], indent=2),
    )

    response = client.chat.completions.create(
        model=EVAL_MODEL_NAME,
        temperature=JUDGE_TEMP,
        max_tokens=JUDGE_MAX_TOKENS,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_WITH_SCHEMA},
            {"role": "user",   "content": user_message},
        ],
    )

    return json.loads(response.choices[0].message.content)

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to evaluate the zero-shot report using the LLM judge and display its faithfulness and analytical quality scores with the corresponding reasoning.


In [ ]:
# Judge the Zero-Shot report
print("Running LLM Judge on Zero-Shot report...")
zero_shot_judge = run_llm_judge(client_document, manufacturing_document, zero_shot_result)

print("\nZERO-SHOT JUDGE SCORES:")
print("-" * 60)
print(f"  Faithfulness       : {zero_shot_judge['faithfulness']['score']} / 5")
print(f"  Analytical Quality : {zero_shot_judge['analytical_quality']['score']} / 5")
print("\nFaithfulness reasoning:")
print(zero_shot_judge['faithfulness']['reasoning'])
print("\nAnalytical quality reasoning:")
print(zero_shot_judge['analytical_quality']['reasoning'])

Running LLM Judge on Zero-Shot report...

ZERO-SHOT JUDGE SCORES:
------------------------------------------------------------
  Faithfulness       : 4 / 5
  Analytical Quality : 3 / 5

Faithfulness reasoning:
The report accurately reflects the information from the source documents without invention or distortion. However, it omits the need for FDA Class III approval, which is a critical regulatory requirement for the LVAD.

Analytical quality reasoning:
The report effectively identifies and analyzes the gaps and necessary adjustments for LVAD production, providing useful recommendations. However, it fails to prioritize the critical regulatory requirement of FDA Class III approval and lacks detailed analysis on achieving ISO 14708-5 compliance, which are essential for decision-making.


**Observations**

* The Zero-Shot report achieves a **Faithfulness score of 4/5** and an **Analytical Quality score of 3/5**, indicating that it is generally accurate but has room for improvement in its depth of analysis.
* The judge finds the report **well grounded in the source documents**, but notes that it omits the **FDA Class III approval requirement**, a critical regulatory consideration for the proposed LVAD.
* While the analysis identifies the major manufacturing gaps and recommends necessary adjustments, it **does not prioritize key regulatory requirements or provide sufficient detail on achieving ISO 14708-5 compliance**, limiting its usefulness for informed decision-making.


# **Report Evaluation using LLM-as-Judge - CoT Report**

The Chain-of-Thought report is evaluated using the **same judge, source documents, evaluation rubric, and scoring criteria** as the Zero-Shot report. This ensures that any difference in scores reflects the impact of the prompting technique rather than changes in the evaluation process.


## **Generate the CoT Judge Scores**

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to evaluate the Chain-of-Thought (CoT) report using the LLM judge and display its faithfulness and analytical quality scores with the corresponding reasoning.


evaluation remains same - generation will changes beased on methord

In [ ]:
# Reuse the same run_llm_judge function - only the report input changes
print("Running LLM Judge on CoT report...")
cot_judge = run_llm_judge(client_document, manufacturing_document, cot_result)

print("\nCoT JUDGE SCORES:")
print("-" * 60)
print(f"  Faithfulness       : {cot_judge['faithfulness']['score']} / 5")
print(f"  Analytical Quality : {cot_judge['analytical_quality']['score']} / 5")
print("\nFaithfulness reasoning:")
print(cot_judge['faithfulness']['reasoning'])
print("\nAnalytical quality reasoning:")
print(cot_judge['analytical_quality']['reasoning'])

Running LLM Judge on CoT report...

CoT JUDGE SCORES:
------------------------------------------------------------
  Faithfulness       : 5 / 5
  Analytical Quality : 4 / 5

Faithfulness reasoning:
The report accurately reflects the capabilities and limitations of the manufacturer based on the provided pacemaker documentation and the proposed LVAD requirements.

Analytical quality reasoning:
The report effectively identifies and prioritizes the key gaps and limitations in the manufacturer's capabilities, providing clear and actionable recommendations. However, it could improve by offering more detailed insights into the cost and time required for the proposed adjustments.


**Observations**

* The CoT report achieves a **perfect Faithfulness score (5/5)**, with the judge confirming that all claims are supported by the source documents.
* It maintains a strong **Analytical Quality score (4/5)** by clearly prioritizing critical production gaps and providing actionable recommendations, particularly around pump manufacturing and control systems.
* The remaining improvement area is **more detailed compliance guidance**, indicating that while the reasoning is stronger than the Zero-Shot report, regulatory recommendations could be further expanded.


# **Save the Final Report**

The Zero-Shot and CoT reports are first compared using their evaluation scores. The selected feasibility report, along with the judge evaluations and score comparison, is then saved as the final report artifact for review.

## **Side-by-Side Score Comparison**

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to create a side-by-side scoreboard comparing the Zero-Shot and Chain-of-Thought (CoT) evaluation scores for faithfulness and analytical quality.


In [ ]:
# Build a compact scoreboard so the two branches can be compared at a glance
# This is a simple side-by-side view of the judge scores from both branches

scoreboard = {
    "Zero-Shot": {
        "Faithfulness":       zero_shot_judge["faithfulness"]["score"],
        "Analytical Quality": zero_shot_judge["analytical_quality"]["score"],
    },
    "CoT": {
        "Faithfulness":       cot_judge["faithfulness"]["score"],
        "Analytical Quality": cot_judge["analytical_quality"]["score"],
    },
}

print("BRANCH-VS-BRANCH SCOREBOARD")
print("=" * 60)
print(f"{'Metric':<25} {'Zero-Shot':>12} {'CoT':>12}")
print("-" * 60)
for metric in ["Faithfulness", "Analytical Quality"]:
    zs = scoreboard["Zero-Shot"][metric]
    ct = scoreboard["CoT"][metric]
    print(f"{metric:<25} {zs:>12} / 5 {ct:>7} / 5")

BRANCH-VS-BRANCH SCOREBOARD
Metric                       Zero-Shot          CoT
------------------------------------------------------------
Faithfulness                         4 / 5       5 / 5
Analytical Quality                   3 / 5       4 / 5


**Observation**

* The **CoT report** outperforms the **Zero-Shot report** across both evaluation metrics, achieving **5/5 for Faithfulness** and **4/5 for Analytical Quality**, compared to **4/5** and **3/5**, respectively.
* The higher **Faithfulness** score indicates that the CoT report aligns more closely with the source documents and captures important details that were missed in the Zero-Shot analysis.
* The improvement in **Analytical Quality** suggests that the CoT report provides a more comprehensive and decision-oriented assessment, particularly by giving greater attention to critical regulatory requirements and compliance considerations.



> **Note:** The evaluation scores may vary across runs because we are not using a rubric-based evaluation. Instead, the > LLM determines its own evaluation criteria for each assessment, which can lead to slight differences in scoring even > when evaluating the same report.

> In the upcoming weeks, we will dive deeper into evaluating LLM outputs and building robust Generative AI workflows. You will learn how to design rubric-based evaluations, define consistent scoring criteria, and systematically assess the quality, faithfulness, and reliability of LLM-generated responses.


## **Assemble and Save the Final Report**

The **Zero-Shot** and **Chain-of-Thought (CoT)** reports are compared using their judge scores. Since the **CoT report** achieves the higher overall evaluation, it is selected as the **Final Report**.

***Prompt***:

<font size=3 color="#4682B4"><b>Write Python code to save the Chain-of-Thought (CoT) feasibility report as a formatted JSON text file and print the file path.


In [ ]:
COT_REPORT_PATH = "feasibility_report_cot.txt"

with open(COT_REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(json.dumps(cot_result["feasibility_report"], indent=2))

print(f"CoT report saved to: {COT_REPORT_PATH}")

CoT report saved to: feasibility_report_cot.txt


# **Business Insights & Recommendations**

## Business Insights



1. **Zero-Shot prompting** produces a valid, structured feasibility report with minimal prompting, making it a strong baseline.
2. **Chain-of-Thought (CoT)** provides deeper analysis by identifying critical production gaps, prioritizing risks, and generating more actionable recommendations.
3. The **CoT report achieves higher faithfulness (5/5 vs. 4/5)** and **higher analytical quality (4/5 vs. 3/5)**, showing better alignment with the source documents and stronger depth of analysis.
4. Using an **LLM-as-Judge** enables objective evaluation through independent Faithfulness and Analytical Quality scores.
5. The combined workflow produces both a feasibility report and an evaluation trail, making the results easier to review and compare.


## Recommendations

1. **Adopt Chain-of-Thought prompting** for production, as it provides more faithful and comprehensive feasibility assessments.
2. **Retain Zero-Shot prompting** as a lightweight baseline for benchmarking future prompt improvements.
3. **Include LLM judge evaluations** with every generated report to provide an auditable measure of report quality.
4. **Evaluate the pipeline on additional document pairs** to validate that the observed improvements generalize across scenarios.
5. **Track Faithfulness and Analytical Quality scores over time** to monitor report quality and identify regressions in prompts or model performance.



<font size=6 color='#4682B4'>Power Ahead!</font>
